In [ ]:
import re
import time
from os import listdir, walk, makedirs, remove
from os.path import isdir, join, exists
import pandas as pd
import swifter
from simplet5 import SimpleT5

In [ ]:
experiment_code = 'A'
version = 1
fold = 2
k = 5
experiment_name = 'experimento-A-overfitting'
model_name = 't5-small'
dataset_path = './dataset'
model_path = '../models/' + model_name

In [ ]:
dataset_basename = '{}-03-preparing-dataset-pt-{}'.format(experiment_code, fold)
train_path = '{}/{}-train.csv'.format(dataset_path, dataset_basename)
val_path = '{}/{}-val.csv'.format(dataset_path, dataset_basename)
print('train path: ' + train_path)
print('val path: ' + val_path)

basename = '04-{}-v{}-{}_{}-{}-{}'.format(experiment_code, version, fold, k, experiment_name, model_name)
models_path = '{}-1-training-outputs'.format(basename)
output_path = '{}-2-accuracy-predictions'.format(basename)
accuracies_path = '{}-2-accuracy-predictions.csv'.format(basename)
print('models path: ' + models_path)
print('output path: ' + output_path)
print('accuracies path: ' + accuracies_path)

In [ ]:
if not exists('{}'.format(output_path)):
    makedirs('{}'.format(output_path))

In [ ]:
if exists(accuracies_path):
    remove(accuracies_path)

In [ ]:
df_val = pd.read_csv(val_path)
df_val['source_text'] = df_val['source_text'].astype(str)
df_val['target_text'] = df_val['target_text'].astype(str)

In [ ]:
df_val.head()

In [ ]:
test = df_val.copy()
test['target_text'].value_counts()

In [ ]:
def get_models():
    models = [join(models_path, name) for name in listdir(models_path) if isdir(join(models_path, name))]
    models.sort()
    return models

In [ ]:
df_accuracies = pd.DataFrame(columns=['experiment', 'epoch', 'accuracy', 'train_loss', 'val_loss'])

start = time.time()

train_losses = []
val_losses = []
epochs = []
accuracies = []

i = 0
m = get_models()
while i < len(m):
    model = m[i]
    print(model)
    
    model_train_data = SimpleT5()
    model_train_data.load_model('t5', model, use_gpu = False)
    test['prediction'] = test['source_text'].swifter.apply(lambda x: model_train_data.predict(x)[0])
    
    epoch_data = re.findall("simplet5\-epoch\-(\d+)\-train\-loss\-(\d+\.\d+)\-val\-loss\-(\d+\.\d+)", model)[0]
    train_losses.append(float(epoch_data[1]))
    val_losses.append(float(epoch_data[2]))
    epochs.append(int(epoch_data[0]))
    accuracy = test[test['prediction']==test['target_text']].shape[0]/test.shape[0]
    accuracies.append(accuracy)
    
    test.to_csv('{}/epoch-{}.csv'.format(output_path, epoch_data[0]), index=False)
    test = test.drop('prediction', axis=1)
    
    if exists(accuracies_path):
        df_accuracies = pd.read_csv(accuracies_path, sep = ';', decimal= ',')
    df_accuracies.loc[int(epoch_data[0])] = [basename, int(epoch_data[0]), accuracy, float(epoch_data[1]), float(epoch_data[2])]
    df_accuracies.to_csv(accuracies_path, index=False, sep = ';', decimal= ',', float_format='%.15f')

    i = i + 1
    m = get_models()

end = time.time()
print('Time: {} rows in {}s'.format(test.shape[0], end - start))

In [ ]:
print('train_losses:', train_losses)
print('val_losses:', val_losses)
print('epochs:', epochs)
print('accuracies:', accuracies)

In [ ]:
df = pd.DataFrame({
   'train_loss': train_losses,
   'val_loss': val_losses
   }, index=range(len(train_losses)))
lines = df.plot.line(style='.-', figsize=(10, 7))
lines.set_title('Loss Chart ({})'.format(model_name),fontsize= 20)
lines.set_xlabel("epoch", fontsize=14)
lines.set_ylabel("loss", fontsize=14)
lines.set_xticks(range(len(train_losses)))

In [ ]:
df = pd.DataFrame({
   'accuracies': accuracies
   }, index=range(len(train_losses)))
lines = df.plot.line(style='.-', figsize=(10, 7))
lines.set_title('Accuracy Chart ({})'.format(model_name),fontsize= 20)
lines.set_xlabel("epoch", fontsize=14)
lines.set_ylabel("accuracy", fontsize=14)
lines.set_xticks(range(len(train_losses)))